# BKK-Pulse: Stage 03 Multi-Expert YOLO Batch Detection (Google Colab A100)

Notebook นี้ใช้รัน pipeline ตรวจจับยานพาหนะแบบครบทั้ง `60,029` รูป (`100` กล้อง) บน Google Colab GPU (แนะนำ **A100** หรือ L4/T4) เพื่อสร้างผลลัพธ์ตาม `DATA_CONTRACT.md`:
1. `data/vehicle_counts.parquet` (`60,029` แถว ครบ 100 กล้อง) + `data/vehicle_counts_sample.parquet` (`2,929` แถว 5 กล้องนำร่อง)
2. รูปที่ตีกรอบ bounding box + ROI + banner สรุปจำนวนรถ (`car`, `motorcycle`, `bus`, `truck`) แพ็กเป็นไฟล์ `.zip` แยกชุดพร้อมโหลด

---

### สิ่งที่ต้องเตรียมใน `.zip` ก่อนอัปโหลด (~2.3 GB)
รวมโฟลเดอร์/ไฟล์ต่อไปนี้จากโปรเจกต์ `Data_Science_Project`:
- `03_cv_yolo_pipeline/` (โค้ด `robust_detector.py`, `run_batch_cv_pipeline.py`, ฯลฯ)
- `Dataset_Clean_V1/` (`60,029` รูป `.jpg` + `image_quality_manifest.parquet`)
- `models/` (ไฟล์ weight `.pt` ทั้ง 6 ตัว; **ไม่ต้องใส่** `sam2.1_b.pt` เพื่อลดขนาดลง ~154 MB)
- `data/camera_rois.json` และ `data/median_backgrounds/` (200 รูปพื้นหลัง)
- `run_pipeline.py` และ `DATA_CONTRACT.md`

> **คำแนะนำการอัปโหลด:** ถ้าไฟล์ `.zip` ใหญ่เกิน ~2 GB การอัปโหลดขึ้น **Google Drive** ก่อนแล้ว mount Drive ใน Cell 2 จะเสถียรกว่าการอัปโหลดผ่านแถบข้างของ Colab โดยตรง (แต่ Notebook นี้รองรับทั้งสองแบบอัตโนมัติครับ)


In [ ]:
#@title 1. ตรวจสอบ GPU (A100 / L4 / T4), CPU และพื้นที่ SSD ใน Colab
import os
import shutil
import subprocess
import torch

print(subprocess.check_output(["nvidia-smi"], text=True))
assert torch.cuda.is_available(), "CUDA GPU not detected! Please select Runtime -> Change runtime type -> GPU (A100)."

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
cpu_cnt = os.cpu_count() or 1
disk_total, disk_used, disk_free = shutil.disk_usage("/content")

print(f"GPU       : {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"CPU cores : {cpu_cnt}")
print(f"SSD /content free: {disk_free / (1024 ** 3):.1f} GB / {disk_total / (1024 ** 3):.1f} GB")

if vram_gb < 14:
    print("[WARN] VRAM < 14 GB -> แนะนำตั้ง BATCH_FRAMES=8, YOLO_BATCH=32 ใน Cell 2")
elif vram_gb < 35:
    print("[INFO] L4/T4 class GPU -> แนะนำ BATCH_FRAMES=16, YOLO_BATCH=48")
else:
    print("[INFO] A100 class GPU detected -> พร้อมรัน BATCH_FRAMES=24, YOLO_BATCH=64 เต็มสปีด")


In [ ]:
#@title 2. เชื่อมต่อ Google Drive (สำหรับอ่าน .zip / เซฟ Checkpoint & ผลลัพธ์กันหลุด) และตั้งค่าพารามิเตอร์
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# === USER CONFIGURATION ===
# ระบุ path ของไฟล์ .zip หรือตั้ง "AUTO" ให้สแกนหาใน /content/*.zip และ /content/drive/MyDrive/*.zip
ZIP_PATH = "AUTO"

# โฟลเดอร์บน Google Drive สำหรับเก็บ checkpoints (รายกล้อง) และไฟล์ผลลัพธ์สุดท้าย
DRIVE_OUT_DIR = "/content/drive/MyDrive/BKK_Pulse_Colab_Outputs"
WORK_EXTRACT_DIR = "/content/bkk_workspace"

# โหมดการรัน: "all" (100 กล้อง / 60,029 รูป) | "sample" (5 กล้องนำร่อง) | "smoke" (เทสต์ 6 รูปแรก)
RUN_MODE = "all"

# จูนความเร็วสำหรับ A100 (40GB/80GB):
BATCH_FRAMES = 24     # จำนวนเฟรมต่อรอบ CPU+GPU batch (A100: 24-32 | T4/L4: 12-16)
YOLO_BATCH = 64       # จำนวน crops สูงสุดต่อ 1 YOLO forward pass (A100: 64-96 | T4: 32)
CPU_WORKERS = 8       # Threads สำหรับอ่าน/เตรียมภาพ & เขียนรูปตีกรอบ
CONF_THRESH = 0.25

# บันทึกรูปที่ตีกรอบแล้ว (เขียนลง /content SSD ก่อน แล้วค่อยแพ็ก .zip ส่งเข้า Drive ใน Cell 7)
SAVE_ANNOTATED_IMAGES = True
IMAGE_SCALE = 1.5     # 1.5 ภาพคมชัดอ่านง่าย (~4-5 GB ทั้งหมด); 1.0 (~2-3 GB); 2.0 (~7.7 GB)
ANNOTATED_DIR = "/content/annotated_frames"
ZIP_CAMERAS_PER_ARCHIVE = 20  # แบ่ง .zip ละ ~20 กล้อง (~1 GB/ไฟล์) เพื่อให้โหลดจาก Drive ง่ายไม่ขาดกลางคัน


In [ ]:
#@title 3. แตกไฟล์ .zip ลง Colab NVMe SSD (/content) และตรวจสอบความครบของ Asset
import os
from pathlib import Path
import shutil
import subprocess

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
os.makedirs(WORK_EXTRACT_DIR, exist_ok=True)


def find_project_root(search_root: Path) -> Path | None:
    hits = sorted(search_root.rglob("run_batch_cv_pipeline.py"))
    for h in hits:
        if len(h.parents) >= 3:
            return h.parents[2]
    return None


def locate_zip_file(zip_cfg: str) -> Path:
    if zip_cfg != "AUTO":
        p = Path(zip_cfg)
        if not p.is_file():
            raise FileNotFoundError(f"ไม่พบไฟล์ zip ตาม ZIP_PATH ที่ระบุ: {p}")
        return p

    search_dirs = [Path("/content"), Path("/content/drive/MyDrive")]
    candidates = []
    for d in search_dirs:
        if d.is_dir():
            candidates.extend(sorted(d.glob("*.zip"), key=lambda x: x.stat().st_size, reverse=True))
            # Check one level deep in MyDrive (e.g. MyDrive/DSDE/*.zip)
            if "drive" in str(d):
                for sub in d.iterdir():
                    if sub.is_dir():
                        candidates.extend(sorted(sub.glob("*.zip"), key=lambda x: x.stat().st_size, reverse=True))

    if not candidates:
        raise FileNotFoundError(
            "ไม่พบไฟล์ .zip ใน /content หรือ /content/drive/MyDrive/ ! กรุณาอัปโหลดไฟล์ .zip หรือระบุ path ใน ZIP_PATH"
        )
    print("พบไฟล์ .zip ที่พร้อมใช้งาน:")
    for i, c in enumerate(candidates[:5]):
        print(f"  [{i}] {c} ({c.stat().st_size / (1024 ** 3):.2f} GB)")
    return candidates[0]


PROJECT_ROOT = find_project_root(Path(WORK_EXTRACT_DIR))
if PROJECT_ROOT is None:
    zip_file = locate_zip_file(ZIP_PATH)
    print(f"กำลังแตกไฟล์จาก: {zip_file} -> {WORK_EXTRACT_DIR} ...")
    # Prefer native unzip binary (multi-times faster than Python zipfile for 60,000+ small files)
    if shutil.which("unzip"):
        res = subprocess.run(["unzip", "-q", "-o", str(zip_file), "-d", WORK_EXTRACT_DIR], check=False)
        if res.returncode not in (0, 1):
            raise RuntimeError(f"unzip failed with exit code {res.returncode}")
    else:
        import zipfile
        with zipfile.ZipFile(zip_file, "r") as zf:
            zf.extractall(WORK_EXTRACT_DIR)
    PROJECT_ROOT = find_project_root(Path(WORK_EXTRACT_DIR))

assert PROJECT_ROOT is not None, f"ไม่พบ 03_cv_yolo_pipeline/src/run_batch_cv_pipeline.py ภายใน {WORK_EXTRACT_DIR}"
os.chdir(PROJECT_ROOT)
print(f"\n[OK] PROJECT_ROOT = {PROJECT_ROOT}")

# Verify essential assets
manifest_path = PROJECT_ROOT / "Dataset_Clean_V1" / "image_quality_manifest.parquet"
rois_path = PROJECT_ROOT / "data" / "camera_rois.json"
bg_dir = PROJECT_ROOT / "data" / "median_backgrounds"
models_dir = PROJECT_ROOT / "models"

assert manifest_path.is_file(), f"Missing {manifest_path}"
assert rois_path.is_file(), f"Missing {rois_path}"
bg_count = len(list(bg_dir.glob("*.jpg"))) if bg_dir.is_dir() else 0
pt_files = list(models_dir.rglob("*.pt")) if models_dir.is_dir() else []

print(f"Manifest     : OK ({manifest_path})")
print(f"Camera ROIs  : OK ({rois_path})")
print(f"Median BGs   : {bg_count} .jpg files cached")
print(f"Model .pt    : {len(pt_files)} files found locally" + (" (missing models will auto-download)" if len(pt_files) < 6 else " (all cached offline)"))


In [ ]:
#@title 4. ติดตั้งแพ็กเกจเพิ่มเติม (`ultralytics`) และทดสอบโหลด Detector
!pip install -q ultralytics pyarrow

import sys
import pandas as pd
import pyarrow

sys.path.insert(0, str(PROJECT_ROOT / "03_cv_yolo_pipeline" / "src"))
from robust_detector import RobustTrafficDetector, CANONICAL_ID_TO_NAME

df_manifest = pd.read_parquet(PROJECT_ROOT / "Dataset_Clean_V1" / "image_quality_manifest.parquet", columns=["cam_id", "is_valid", "quality_warning"])
valid_df = df_manifest[df_manifest["is_valid"] & (df_manifest["quality_warning"] == "none")]
print(f"[OK] Ready! Valid images: {len(valid_df):,} across {valid_df['cam_id'].nunique()} cameras")
print(f"Canonical classes: {CANONICAL_ID_TO_NAME}")


In [ ]:
#@title 5. (Optional) ทดสอบ Smoke Test 6 รูป + วัดความเร็ว A100 จริง + ดูตัวอย่างภาพตีกรอบ
import glob
import subprocess
from IPython.display import Image as IPyImage, display

smoke_parquet = "/content/smoke_vehicle_counts.parquet"
smoke_imgs = "/content/smoke_annotated"

smoke_cmd = [
    "python", "-u", "03_cv_yolo_pipeline/src/run_batch_cv_pipeline.py",
    "--cam-ids", "370", "395",
    "--limit", "6",
    "--batch-frames", "6",
    "--yolo-batch", str(YOLO_BATCH),
    "--workers", str(CPU_WORKERS),
    "--conf", str(CONF_THRESH),
    "--output-parquet", smoke_parquet,
    "--checkpoint-dir", "/content/smoke_ckpts",
    "--no-resume",
    "--save-images",
    "--images-dir", smoke_imgs,
    "--save-scale", str(IMAGE_SCALE),
]
subprocess.run(smoke_cmd, cwd=str(PROJECT_ROOT), check=True)

sample_jpgs = sorted(glob.glob(f"{smoke_imgs}/**/*.jpg", recursive=True))
print(f"\nSaved {len(sample_jpgs)} smoke preview image(s). Showing first & last:")
for p in (sample_jpgs[:1] + sample_jpgs[-1:]):
    print(" ->", p)
    display(IPyImage(filename=p))


In [ ]:
#@title 6. รัน Batch Detection Pipeline ครบทุกภาพ (เซฟ Checkpoint รายกล้องลง Drive อัตโนมัติ)
import os
from pathlib import Path
import shlex
import shutil
import subprocess

ckpt_dir = Path(DRIVE_OUT_DIR) / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

mode_args = ["--all"]
if RUN_MODE == "sample":
    mode_args = ["--sample"]
elif RUN_MODE == "smoke":
    mode_args = ["--cam-ids", "370", "395", "--limit", "10"]

cmd_args = [
    "python", "-u", "03_cv_yolo_pipeline/src/run_batch_cv_pipeline.py",
    *mode_args,
    "--batch-frames", str(BATCH_FRAMES),
    "--yolo-batch", str(YOLO_BATCH),
    "--workers", str(CPU_WORKERS),
    "--conf", str(CONF_THRESH),
    "--checkpoint-dir", str(ckpt_dir),
]
if SAVE_ANNOTATED_IMAGES:
    cmd_args.extend(["--save-images", "--images-dir", str(ANNOTATED_DIR), "--save-scale", str(IMAGE_SCALE)])

print("Executing:", " ".join(shlex.quote(a) for a in cmd_args), "\n")
proc = subprocess.Popen(
    cmd_args,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Batch pipeline exited with code {ret}")

# Copy canonical parquet results safely to Google Drive & verify contract
for fname in ("vehicle_counts.parquet", "vehicle_counts_sample.parquet"):
    src = PROJECT_ROOT / "data" / fname
    if src.is_file():
        dst = Path(DRIVE_OUT_DIR) / fname
        shutil.copy2(src, dst)
        print(f"[DRIVE SYNC] {src} -> {dst} ({dst.stat().st_size / (1024 ** 2):.2f} MB)")

print("\nRunning DATA_CONTRACT verification...")
subprocess.run(["python", "run_pipeline.py", "--check-contract"], cwd=str(PROJECT_ROOT), check=True)


In [ ]:
#@title 7. แพ็กภาพที่ตีกรอบแล้วเป็น .zip ส่งเข้า Google Drive (แบ่งไฟล์ละ ~20 กล้อง โหลดง่ายไม่พัง)
import os
from pathlib import Path
import shutil
import time
import zipfile

if SAVE_ANNOTATED_IMAGES and Path(ANNOTATED_DIR).is_dir():
    img_root = Path(ANNOTATED_DIR)
    cam_dirs = sorted(
        [d for d in img_root.iterdir() if d.is_dir() and d.name.isdigit()],
        key=lambda p: int(p.name),
    )
    print(f"Found {len(cam_dirs)} camera folders in {img_root}")

    if cam_dirs:
        chunk_size = max(1, int(ZIP_CAMERAS_PER_ARCHIVE))
        out_dir = Path(DRIVE_OUT_DIR)
        out_dir.mkdir(parents=True, exist_ok=True)

        for idx in range(0, len(cam_dirs), chunk_size):
            part_dirs = cam_dirs[idx : idx + chunk_size]
            first_cam = part_dirs[0].name
            last_cam = part_dirs[-1].name
            part_no = (idx // chunk_size) + 1
            zip_name = f"annotated_frames_part{part_no:02d}_cam{first_cam}_to_{last_cam}.zip"
            tmp_zip = Path("/content") / zip_name
            dst_zip = out_dir / zip_name

            t_z = time.perf_counter()
            n_imgs = 0
            # ZIP_STORED avoids re-compressing already-compressed JPEGs (~10x faster)
            with zipfile.ZipFile(tmp_zip, mode="w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
                for cdir in part_dirs:
                    for jpg in sorted(cdir.glob("*.jpg")):
                        zf.write(jpg, arcname=f"{cdir.name}/{jpg.name}")
                        n_imgs += 1

            shutil.copy2(tmp_zip, dst_zip)
            size_mb = dst_zip.stat().st_size / (1024 ** 2)
            tmp_zip.unlink(missing_ok=True)
            print(
                f"[ZIP {part_no:02d}] {dst_zip.name} -> {n_imgs:,} images, "
                f"{size_mb:.1f} MB ({time.perf_counter() - t_z:.1f}s)"
            )
else:
    print("Skipped annotated image archiving (SAVE_ANNOTATED_IMAGES is False or folder empty).")

print("\n=== Summary of Files Saved in Google Drive ===")
for f in sorted(Path(DRIVE_OUT_DIR).glob("*")):
    if f.is_file():
        print(f" - {f.name} ({f.stat().st_size / (1024 ** 2):.2f} MB)")


In [ ]:
#@title 8. (Optional) ดาวน์โหลด vehicle_counts.parquet ลงเครื่องผ่านเบราว์เซอร์ทันที
from google.colab import files

for fname in ("vehicle_counts.parquet", "vehicle_counts_sample.parquet"):
    p = PROJECT_ROOT / "data" / fname
    if p.is_file():
        print(f"Downloading {p.name}...")
        files.download(str(p))
